In [1]:
import simpy
import random
import numpy as np

MEAN_ARRIVAL_TIME = 2.4 / 60   # Fixed at 2.4 minutes (Unit: Hours)
SIM_TIME = 720                 # 720 hours = 1 month
NUM_REPLICATIONS = 100         # Fixed 100 replications for Monte Carlo

# Initial CAPEX and Fixed Costs (Monthly basis)
COST_ULTRA_TOTAL = 200_000_000 
COST_FAST_TOTAL = 65_000_000 
DEPRECIATION_MONTHS = 60 

COST_ULTRA_MONTHLY = COST_ULTRA_TOTAL / DEPRECIATION_MONTHS
COST_FAST_MONTHLY = COST_FAST_TOTAL / DEPRECIATION_MONTHS

# Detailed monthly OPEX per charger
OP_COST_ULTRA = 953_000  
OP_COST_FAST = 300_800  

class EVChargingStation:
    """
    Represents the Electric Vehicle (EV) Charging Station environment.
    Manages the charging resources (Ultra-fast and Fast chargers), 
    system capacity constraints (K), and tracks key performance metrics 
    such as waiting times, revenue, and customer loss (Balking/Blocking).
    """
    def __init__(self, env, num_ultra=4, num_fast=4):
        """
        Initialize the charging station with a specific infrastructure setup.
        
        Args:
            env (simpy.Environment): The SimPy simulation environment controlling the time.
            num_ultra (int): Initial number of Ultra-fast chargers (Default: 4).
            num_fast (int): Initial number of Fast chargers (Default: 4).
        """
        self.env = env
        self.num_ultra = num_ultra
        self.num_fast = num_fast
        self.ultra_fast = simpy.PriorityResource(env, capacity=num_ultra)
        self.fast = simpy.PriorityResource(env, capacity=num_fast)
        
        self.n_a = 0
        self.n_balking = 0
        self.n_blocking = 0
        self.c_ultra = 0
        self.c_fast = 0
        
        self.K = 20
        
        self.revenue = 0
        self.n_ticket = 0
        self.n_served = 0
        self.wait_times = []
        self.total_times = []
        
        # Variables to track total busy time for utilization (ρ) calculation
        self.busy_time_ultra = 0.0
        self.busy_time_fast = 0.0

    def calculate_cost(self):
        """
        Calculate the total monthly cost (CAPEX + OPEX) of the station.
        Capital Expenditure (Depreciation) is applied only to newly added chargers (delta), 
        while Operational Expenditure is applied to all active chargers.
        
        Returns:
            float: Total expected monthly cost in KRW.
        """
        delta_ultra = max(0, self.num_ultra - 4)
        delta_fast = max(0, self.num_fast - 4)
        
        # CAPEX (Depreciation for added chargers) + OPEX (for all active chargers)
        depreciation_cost = (COST_ULTRA_MONTHLY * delta_ultra) + (COST_FAST_MONTHLY * delta_fast)
        operation_cost = (OP_COST_ULTRA * self.num_ultra) + (OP_COST_FAST * self.num_fast)
        
        return depreciation_cost + operation_cost

In [2]:
def calculate_ewi(resource, mu):
    """
    Calculate the Estimated Waiting Time (EWI) for an incoming EV.
    This metric drives the state-dependent customer behavior (e.g., Balking).
    
    Args:
        resource (simpy.PriorityResource): The target charging station (Ultra-fast or Fast).
        mu (float): Average service rate (vehicles processed per unit time).
        
    Returns:
        float: Expected waiting time. Returns infinity if capacity or mu is 0.
    """
    n = len(resource.queue) + resource.count  # Total vehicles in system (Queue + Charging)
    c = resource.capacity                     # Total number of active servers
    
    if c == 0 or mu == 0:
        return float('inf')
    return n / (c * mu)

def generate_car_attributes():
    """
    Generate probabilistic attributes for each arriving EV using Monte Carlo methods.
    
    Returns:
        tuple: (Battery Capacity, Initial SoC, Emergency Flag, Pre-purchase Flag)
    """
    rand_v = random.random()
    
    # 1. Assign Battery Capacity (cap_v) based on assumed market distribution
    if rand_v < 0.20:
        cap_v = 50   # e.g., Compact EVs
    elif rand_v < 0.80:
        cap_v = 77   # e.g., Standard Sedans/SUVs (Ioniq 5, EV6)
    else:
        cap_v = 100  # e.g., Heavy-duty Trucks or Premium EVs

    # 2. Initial State of Charge (SoC) using Beta Distribution (Right-skewed)
    battery_first = np.random.beta(2, 5)
    
    # 3. Emergency Status: True if battery is critically low (<= 10%)
    is_emergency = True if battery_first <= 0.10 else False
    
    # 4. Priority Ticket: 20% baseline probability of having a pre-purchased ticket
    pre_purchase = True if random.random() <= 0.20 else False
    
    return cap_v, battery_first, is_emergency, pre_purchase

In [6]:
def car_process(env, car_id, station):
    """
    Generator function representing the lifecycle of a single EV customer in the simulation.
    Handles routing, queueing discipline, balking/blocking evaluation, and revenue calculation.
    """
    station.n_a += 1
    arrival_time = env.now
    
    # Generate randomized EV and driver attributes
    cap_v, battery_first, is_emergency, pre_purchase = generate_car_attributes()
    
    # 1. Calculate Estimated Waiting Time (EWI) for both server groups
    ewi_fast = calculate_ewi(station.fast, 2.30) 
    ewi_ultra = calculate_ewi(station.ultra_fast, 7.86)
    
    # Determine the minimum expected waiting time (best route)
    best_ewi = min(ewi_fast, ewi_ultra)
    # Customer's patience threshold (Gaussian distribution)
    tolerance_time = max(0.5, random.gauss(1.5, 0.4))
    
    has_ticket = pre_purchase or is_emergency
    
    # 2. Dynamic Priority Ticket Purchase Decision (Modeled via Step Function)
    if not has_ticket:
        t_minutes = best_ewi * 60 
        p2_t = 0.0
        
        if t_minutes >= 30:
            p2_t = 0.25
        elif t_minutes >= 10:
            p2_t = 0.10
            
        if random.random() <= p2_t:
            has_ticket = True

    # 3. Customer Loss Evaluation (Blocking & Balking logic for non-ticket holders)
    if not has_ticket:
        total_cars_in_system = len(station.ultra_fast.queue) + station.ultra_fast.count + \
                               len(station.fast.queue) + station.fast.count
                               
        # Blocking: Involuntary loss due to spatial constraints (K)
        if total_cars_in_system >= station.K:
            station.n_blocking += 1
            return 

        # Balking: Voluntary loss due to EWI exceeding psychological tolerance
        if best_ewi > tolerance_time:
            station.n_balking += 1
            return 

    # 4. Apply Queueing Discipline & Initial Revenue Collection
    if has_ticket:
        station.n_ticket += 1
        station.revenue += 3500
        priority = -1  # High priority (Preemptive logic simulated via non-preemptive priority queue)
    else:
        priority = 0   # Standard priority

    # 5. Behavior-Based Routing: Customer chooses the queue with the lowest EWI
    if ewi_ultra <= ewi_fast:
        chosen_server = station.ultra_fast
        power = 350
        efficiency = 0.88
    else:
        chosen_server = station.fast
        power = 100
        efficiency = 0.90

    # 6. Service Request and Execution
    with chosen_server.request(priority=priority) as req:
        yield req 
        
        # Calculate actual waiting time
        start_charge_time = env.now
        wait_time = start_charge_time - arrival_time
        station.wait_times.append(wait_time)
        
        # Calculate charging service time based on physical constraints
        charge_amount = cap_v * max(0, 0.8 - battery_first)
        charge_time = charge_amount / (power * efficiency)
        
        # Calculate electricity margin (KRW per kWh)
        if power == 350:  
            margin_per_kwh = 494.5   
        else:
            margin_per_kwh = 424.16  

        station.revenue += charge_amount * margin_per_kwh
        
        # Track server utilization times
        if chosen_server == station.ultra_fast:
            station.busy_time_ultra += charge_time
        else:
            station.busy_time_fast += charge_time
        
        # Simulate the passing of charging time
        yield env.timeout(charge_time) 
        
        end_time = env.now
        total_time = end_time - arrival_time
        station.total_times.append(total_time) 
        
        # 7. Secondary Revenue Evaluation: F&B sales proportional to sojourn time
        total_time_min = total_time * 60
        snack_prob = random.random()
        
        if total_time_min < 15:
            if snack_prob < 0.15: station.revenue += 2500
            elif snack_prob < 0.20: station.revenue += 6000
        elif 15 <= total_time_min < 40:
            if snack_prob < 0.40: station.revenue += 2500
            elif snack_prob < 0.60: station.revenue += 6000
        else:
            if snack_prob < 0.30: station.revenue += 2500
            elif snack_prob < 0.90: station.revenue += 6000
        
        # Increment service counters
        if chosen_server == station.ultra_fast:
            station.c_ultra += 1
        else:
            station.c_fast += 1
            
        station.n_served += 1

In [4]:
def car_arrival(env, station):
    """
    Generator function representing the arrival process of EVs.
    Currently modeled as a Homogeneous Poisson Process using exponential inter-arrival times.
    """
    car_id = 0
    while True:
        inter_arrival = random.expovariate(1.0 / MEAN_ARRIVAL_TIME)
        yield env.timeout(inter_arrival)
        car_id += 1
        env.process(car_process(env, car_id, station))

# Define the 3 infrastructure expansion scenarios: (Ultra-fast, Fast)
scenarios = [(4, 4), (5, 4), (4, 5)]
results = {}

print(f"⏳ Running {NUM_REPLICATIONS} replications for each of the {len(scenarios)} scenarios...")

for num_u, num_f in scenarios:
    scenario_name = f"Ultra{num_u}_Fast{num_f}"
    
    # Lists to store the outcomes of each replication
    rep_revenues, rep_costs, rep_profits = [], [], []
    rep_loss_rates, rep_wait_times, rep_total_times = [], [], []
    rep_util_ultra, rep_util_fast = [], [] 
    
    for i in range(NUM_REPLICATIONS):
        env = simpy.Environment()
        station = EVChargingStation(env, num_ultra=num_u, num_fast=num_f)
        
        env.process(car_arrival(env, station))
        env.run(until=SIM_TIME)
        
        # Calculate iteration results
        tr = station.revenue
        tc = station.calculate_cost()
        z = tr - tc
        
        loss_rate = (station.n_balking + station.n_blocking) / station.n_a if station.n_a > 0 else 0
        avg_wait = np.mean(station.wait_times) * 60 if station.wait_times else 0
        avg_total = np.mean(station.total_times) * 60 if station.total_times else 0
        
        util_u = (station.busy_time_ultra / (num_u * SIM_TIME)) if num_u > 0 else 0
        util_f = (station.busy_time_fast / (num_f * SIM_TIME)) if num_f > 0 else 0
        
        # Append iteration results
        rep_revenues.append(tr)
        rep_costs.append(tc)
        rep_profits.append(z)
        rep_loss_rates.append(loss_rate)
        rep_wait_times.append(avg_wait)
        rep_total_times.append(avg_total)
        rep_util_ultra.append(util_u)
        rep_util_fast.append(util_f)
        
    # Aggregate and store the mean of all replications
    results[scenario_name] = {
        "Avg Total Revenue (TR)": np.mean(rep_revenues),
        "Avg Total Cost (TC)": np.mean(rep_costs),
        "Avg Profit (Z)": np.mean(rep_profits),
        "Avg Loss Rate (P_Loss)": np.mean(rep_loss_rates),
        "Avg Wait Time": np.mean(rep_wait_times),
        "Avg Total Time": np.mean(rep_total_times),
        "Avg Util Ultra": np.mean(rep_util_ultra),
        "Avg Util Fast": np.mean(rep_util_fast)
    }

# Print Final Aggregated Results
print(f"\n=== Cost-Profit and System Metric Optimization ({NUM_REPLICATIONS} Replications Avg) ===")
for name, data in results.items():
    print(f"\n[ Scenario: {name} ]")
    print(f" - System Avg Wait Time (E[W_q])  : {data['Avg Wait Time']:.2f} min")
    print(f" - System Avg Stay Time (E[W_tot]): {data['Avg Total Time']:.2f} min")
    print(f" - Ultra-fast Utilization (rho_u) : {data['Avg Util Ultra']:.2%}")
    print(f" - Fast Utilization (rho_f)       : {data['Avg Util Fast']:.2%}")
    print(f" - Total Customer Loss Rate       : {data['Avg Loss Rate (P_Loss)']:.5%}")
    print(f" --------------------------------------------------")
    print(f" - Monthly Total Revenue (TR)     : {data['Avg Total Revenue (TR)']:,.0f} KRW")
    print(f" - Monthly Total Cost (TC)        : {data['Avg Total Cost (TC)']:,.0f} KRW")
    print(f" - Monthly Expected Net Profit (Z): {data['Avg Profit (Z)']:,.0f} KRW")

⏳ Running 100 replications for each of the 3 scenarios...

=== Cost-Profit and System Metric Optimization (100 Replications Avg) ===

[ Scenario: Ultra4_Fast4 ]
 - System Avg Wait Time (E[W_q])  : 0.91 min
 - System Avg Stay Time (E[W_tot]): 11.11 min
 - Ultra-fast Utilization (rho_u) : 68.43%
 - Fast Utilization (rho_f)       : 37.88%
 - Total Customer Loss Rate       : 0.00083%
 --------------------------------------------------
 - Monthly Total Revenue (TR)     : 377,606,640 KRW
 - Monthly Total Cost (TC)        : 5,015,200 KRW
 - Monthly Expected Net Profit (Z): 372,591,440 KRW

[ Scenario: Ultra5_Fast4 ]
 - System Avg Wait Time (E[W_q])  : 0.25 min
 - System Avg Stay Time (E[W_tot]): 9.98 min
 - Ultra-fast Utilization (rho_u) : 56.37%
 - Fast Utilization (rho_f)       : 30.77%
 - Total Customer Loss Rate       : 0.00011%
 --------------------------------------------------
 - Monthly Total Revenue (TR)     : 376,787,694 KRW
 - Monthly Total Cost (TC)        : 9,301,533 KRW
 - Month

In [7]:
import os
import pandas as pd

file_name = "ev_simulation_results.xlsx"

# 1. Convert the results dictionary into a pandas DataFrame
df_results = pd.DataFrame(results).T
df_results.index.name = "Scenario"
df_results = df_results.reset_index()

# 2. Check if the file already exists to append data or create a new one
if os.path.exists(file_name):
    # Read existing data
    existing_df = pd.read_excel(file_name)
    # Concatenate new results below the existing data
    combined_df = pd.concat([existing_df, df_results], ignore_index=True)
    # Save the merged data back to the Excel file (overwrite with accumulated data)
    combined_df.to_excel(file_name, index=False)
    print(f" New results successfully appended to '{file_name}'")
else:
    # Create a new file if it does not exist
    df_results.to_excel(file_name, index=False)
    print(f"'{file_name}' created and results saved successfully")

'ev_simulation_results.xlsx' created and results saved successfully
